Dataset Anomalies for Animal Pose Estimation
The ground truth annotations for the bounding boxes and keypoints are only available for single object instances. Besides, there are still quite a few incorrect annotations, as seen from the sample below.

Data anomalies for animal pose estimation
Dataset anomalies
The ground truth annotations for the bounding boxes and keypoints are only available for single object instances. Besides, there are still quite a few incorrect annotations, as seen from the sample below.

As you can see from the top leftmost image, the bounding box and the keypoints have been annotated for two different object instances. The same is reflected in the second and the fourth image in the 1st row (from left to right) and the first and the third image in the 2nd row.

Besides, the keypoints are also incorrectly annotated, as seen in the third image in the first row, where the jaw and the tip of the left ear are incorrectly annotated. The same is true for the first image in the second row, where the tip of the left ear has been mislabeled. Another instance of incorrect annotations has been manifested in the fourth image in the second row, where both ear tips are mislabelled.

As mentioned earlier, there are only single instance annotations for each image. Observe the second image (from the left) in the second row, where we only have annotations for the dog on the left, while there have been three instances of dogs.

Handling Mismatched Ground Truth Annotations across Boxes and Keypoints for Animal Pose Estimation
One intuitive way of handling mismatched box and keypoint annotations is to estimate a rectangle based on the given keypoints. This can be done using the cv2.boundingRect utility function to approximate a rectangle given a set of coordinates. Please pause for a moment and take a look at the samples below.

Handling mismatched annotations for animal pose estimation
Handling mismatched bounding box and keypoint annotations
Although the bounding boxes are imperfect, handling mismatched bounding boxes and keypoint annotations can be inexpensive using the above approach. We can run a detection model such as YOLOv8 to obtain more accurate box annotations and then map the keypoints with the closest bounding box.

However, we shall stick to the annotations provided in the original JSON file for our experiments.

Download Code To easily follow along this tutorial, please download code by clicking on the button below. It's FREE!
Download Code
Creating Annotations Consistent with YOLOv8 for Training and Validation Data
Before we prepare our data, we need to be well-versed in the annotation format for keypoint detection accepted by YOLOv8 pose models from Ultralytics. The following points highlight the dataset format for fine-tuning Ultralytics’ YOLOv8 Pose models:

The dataset format used for training YOLO pose models is as follows:

One text file per image: Each image in the dataset has a corresponding text file with the same name as the image file and the .txt extension.
One row per object: Each row in the text file corresponds to one object instance in the image.
Object information per row: Each row contains the following information about the object instance:
Object class index: An integer representing the object’s class (e.g., 0 for person, 1 for car, etc.).
Object center coordinates: The x and y coordinates of the object’s center are normalized to 0 and 1.
Object width and height: The width and height of the object are normalized to be between 0 and 1.
Object width and height: The width and height of the object are normalized to be between 0 and 1.
Additionally, a visibility flag is associated with the keypoint coordinates. It can contain either of the three values:

 0: Not labeled
 1: Labelled but not visible
 2: Labeled and visible.
The JSON annotations contain an additional boolean visibility flag and the keypoint coordinates discussed earlier. We will set the flags for all the visible keypoints to 2.

Keypoint annotations for fine-tuning pose models in Ultralytics correspond to the following syntax:

<class-index> <xc> <yc> <width> <height> <px1> <py1> <p1-visibility> ... <pxn> <pyn> <pn-visibility>

1
0 0.55991 0.503 0.76688 0.918 0.39143 0.91133 2.0 0.44227 0.72467 2.0
The first item in the entry is the CLASS_ID, followed by the bounding box data (normalized xcenter, ycenter, width, height), and finally the normalized [x y] coordinates along with the visibility flags (i.e., 2 for both keypoints).

In [ ]:
!wget http://vision.stanford.edu/aditya86/ImageNetDogs/images.tar

In [ ]:
!mkdir IMAGES
!tar -xvf images.tar -C IMAGES

In [ ]:
!wget https://github.com/benjiebob/StanfordExtra/raw/master/keypoint_definitions.csv

In [ ]:
!unzip /content/stanfordextra_v12.zip

In [ ]:
import os
import json
ANN_PATH = "StanfordExtra_V12"
JSON_PATH = os.path.join(ANN_PATH, "StanfordExtra_v12.json")

with open(JSON_PATH) as file:
    json_data = json.load(file)

In [ ]:
import numpy as np
train_ids = np.load(os.path.join(ANN_PATH,"train_stanford_StanfordExtra_v12.npy"))
val_ids = np.load(os.path.join(ANN_PATH, "test_stanford_StanfordExtra_v12.npy"))

In [ ]:
DATA_DIR = "animal-pose-data"

TRAIN_DIR         = f"train"
TRAIN_FOLDER_IMG    = f"images"
TRAIN_FOLDER_LABELS = f"labels"

TRAIN_IMG_PATH   = os.path.join(DATA_DIR, TRAIN_DIR, TRAIN_FOLDER_IMG)
TRAIN_LABEL_PATH = os.path.join(DATA_DIR, TRAIN_DIR, TRAIN_FOLDER_LABELS)

VALID_DIR           = f"valid"
VALID_FOLDER_IMG    = f"images"
VALID_FOLDER_LABELS = f"labels"

VALID_IMG_PATH   = os.path.join(DATA_DIR, VALID_DIR, VALID_FOLDER_IMG)
VALID_LABEL_PATH = os.path.join(DATA_DIR, VALID_DIR, VALID_FOLDER_LABELS)

os.makedirs(TRAIN_IMG_PATH, exist_ok=True)
os.makedirs(TRAIN_LABEL_PATH, exist_ok=True)
os.makedirs(VALID_IMG_PATH, exist_ok=True)
os.makedirs(VALID_LABEL_PATH, exist_ok=True)

In [ ]:
train_json_data = []
for train_id in train_ids:
    train_json_data.append(json_data[train_id])

val_json_data = []
for val_id in val_ids:
    val_json_data.append(json_data[val_id])

In [ ]:
from shutil import copyfile
IMAGES_DIR = "/content/IMAGES/Images/"
for data in train_json_data:
    img_file = data["img_path"]
    filename = img_file.split("/")[-1]
    copyfile(os.path.join(IMAGES_DIR, img_file),
             os.path.join(TRAIN_IMG_PATH, filename))


for data in val_json_data:
    img_file = data["img_path"]
    filename = img_file.split("/")[-1]
    copyfile(os.path.join(IMAGES_DIR, img_file),
             os.path.join(VALID_IMG_PATH, filename))

In [ ]:
CLASS_ID = 0

In [ ]:
def create_yolo_boxes_kpts(img_size, boxes, lm_kpts):

    IMG_W, IMG_H = img_size
    # Modify kpts with visibilities as 1s to 2s.
    vis_ones = np.where(lm_kpts[:, -1] == 1.)
    lm_kpts[vis_ones, -1] = 2.

    # Normalizing factor for bboxes and kpts.
    res_box_array = np.array([IMG_W, IMG_H, IMG_W, IMG_H])
    res_lm_array = np.array([IMG_W, IMG_H])

    # Normalize landmarks in the range [0,1].
    norm_kps_per_img = lm_kpts.copy()
    norm_kps_per_img[:, :-1]  = norm_kps_per_img[:, :-1] / res_lm_array

    # Normalize bboxes in the range [0,1].
    norm_bbox_per_img = boxes / res_box_array

    # Create bboxes coordinates to YOLO.
    # x_c, y_c = x_min + bbox_w/2. , y_min + bbox_h/2.
    yolo_boxes = norm_bbox_per_img.copy()
    yolo_boxes[:2] = norm_bbox_per_img[:2] + norm_bbox_per_img[2:]/2.

    return yolo_boxes, norm_kps_per_img

In [ ]:
def create_yolo_txt_files(json_data, LABEL_PATH):

    for data in json_data:

        IMAGE_ID = data["img_path"].split("/")[-1].split(".")[0]

        IMG_WIDTH, IMG_HEIGHT = data["img_width"], data["img_height"]

        landmark_kpts  = np.nan_to_num(np.array(data["joints"], dtype=np.float32))
        landmarks_bboxes = np.array(data["img_bbox"], dtype=np.float32)

        bboxes_yolo, kpts_yolo = create_yolo_boxes_kpts(
                                            (IMG_WIDTH, IMG_HEIGHT),
                                            landmarks_bboxes,
                                            landmark_kpts)

        TXT_FILE = IMAGE_ID+".txt"

        with open(os.path.join(LABEL_PATH, TXT_FILE), "w") as f:

            x_c_norm, y_c_norm, box_width_norm, box_height_norm = round(bboxes_yolo[0],5),\
                                                                  round(bboxes_yolo[1],5),\
                                                                  round(bboxes_yolo[2],5),\
                                                                  round(bboxes_yolo[3],5),\

            kps_flattend = [round(ele,5) for ele in kpts_yolo.flatten().tolist()]
            line = f"{CLASS_ID} {x_c_norm} {y_c_norm} {box_width_norm} {box_height_norm} "
            line+= " ".join(map(str, kps_flattend))
            f.write(line)

In [ ]:
create_yolo_txt_files(train_json_data, TRAIN_LABEL_PATH)
create_yolo_txt_files(val_json_data, VALID_LABEL_PATH)



In [ ]:
import pandas as pd
ann_meta_data = pd.read_csv("keypoint_definitions.csv")
COLORS = ann_meta_data["Hex colour"].values.tolist()

COLORS_RGB_MAP = []
for color in COLORS:
    R, G, B = int(color[:2], 16), int(color[2:4], 16), int(color[4:], 16)
    COLORS_RGB_MAP.append({color: (R,G,B)})

In [ ]:
def draw_landmarks(image, landmarks):

    radius = 5
    # Check if image width is greater than 1000 px.
    # To improve visualization.
    if (image.shape[1] > 1000):
        radius = 8

    for idx, kpt_data in enumerate(landmarks):

        loc_x, loc_y = kpt_data[:2].astype("int").tolist()
        color_id = list(COLORS_RGB_MAP[int(kpt_data[-1])].values())[0]

        cv2.circle(image,
                   (loc_x, loc_y),
                   radius,
                   color=color_id[::-1],
                   thickness=-1,
                   lineType=cv2.LINE_AA)

    return image

In [ ]:
def draw_boxes(image, detections, class_name = "dog", score=None, color=(0,255,0)):

    font_size = 0.25 + 0.07 * min(image.shape[:2]) / 100
    font_size = max(font_size, 0.5)
    font_size = min(font_size, 0.8)
    text_offset = 3

    thickness = 2
    # Check if image width is greater than 1000 px.
    # To improve visualization.
    if (image.shape[1] > 1000):
        thickness = 10

    xmin, ymin, xmax, ymax = detections[:4].astype("int").tolist()
    conf = round(float(detections[-1]),2)
    cv2.rectangle(image,
                  (xmin, ymin),
                  (xmax, ymax),
                  color=(0,255,0),
                  thickness=thickness,
                  lineType=cv2.LINE_AA)

    display_text = f"{class_name}"

    if score is not None:
        display_text+=f": {score:.2f}"

    (text_width, text_height), _ = cv2.getTextSize(display_text,
                                                   cv2.FONT_HERSHEY_SIMPLEX,
                                                   font_size, 2)

    cv2.rectangle(image,
                      (xmin, ymin),
                      (xmin + text_width + text_offset, ymin - text_height - int(15 * font_size)),
                      color=color, thickness=-1)

    image = cv2.putText(
                    image,
                    display_text,
                    (xmin + text_offset, ymin - int(10 * font_size)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    font_size,
                    (0, 0, 0),
                    2, lineType=cv2.LINE_AA,
                )

    return image

In [ ]:
def visualize_annotations(image, box_data, keypoints_data):

    image = image.copy()

    shape_multiplier = np.array(image.shape[:2][::-1]) # (W, H).
    # Final absolute coordinates (xmin, ymin, xmax, ymax).
    denorm_boxes = np.zeros_like(box_data)

    # De-normalize center coordinates from YOLO to (xmin, ymin).
    denorm_boxes[:, :2] = (shape_multiplier/2.) * (2*box_data[:,:2] - box_data[:,2:])

    # De-normalize width and height from YOLO to (xmax, ymax).
    denorm_boxes[:, 2:] = denorm_boxes[:,:2] + box_data[:,2:]*shape_multiplier

    for boxes, kpts in zip(denorm_boxes, keypoints_data):
        # De-normalize landmark coordinates.
        kpts[:, :2]*= shape_multiplier
        image = draw_boxes(image, boxes)
        image = draw_landmarks(image, kpts)

    return image

In [ ]:
from dataclasses import dataclass, field

@dataclass(frozen=True)
class DatasetConfig:
    IMAGE_SIZE:    int   = 640
    BATCH_SIZE:    int   = 16
    CLOSE_MOSAIC:  int   = 10
    MOSAIC:        float = 0.4
    FLIP_LR:       float = 0.0  # Turn off horizontal flip for asymmetric dog keypoints.

@dataclass(frozen=True)
class TrainingConfig:
    DATASET_YAML:   str   = "animal-keypoints.yaml"
    MODEL:          str   = "yolov8m-pose.pt"
    EPOCHS:         int   = 100
    KPT_SHAPE:      tuple = (24, 3)
    PROJECT:        str   = "Animal_Keypoints"
    NAME:           str   = f"{MODEL.split('.')[0]}_{EPOCHS}_epochs"
    CLASSES_DICT:   dict  = field(default_factory=lambda: {0: "dog"})

    # ONNX export settings for browser inference.
    OPSET:          int   = 17
    CONF_THRESHOLD: float = 0.35
    IOU_THRESHOLD:  float = 0.45
    MAX_DET:        int   = 25
    WEB_ONNX_NAME:  str   = "dog-pose.onnx"


In [ ]:
train_config = TrainingConfig()
data_config = DatasetConfig()

## Train the YOLO pose model and export ONNX files

The next cells install the required packages, create the Ultralytics dataset YAML, train the pose model on the generated YOLO keypoint labels, and export both:

1. a **standard/raw ONNX** model, and  
2. a **browser-friendly ONNX** model named **`dog-pose.onnx`** whose output rows match the website parser format:

`[x_center, y_center, width, height, score, 24*3 keypoint values]`


In [ ]:
%pip install -q ultralytics onnx onnxsim

In [ ]:
from pathlib import Path
from types import SimpleNamespace
import yaml
import torch
import onnx

from ultralytics import YOLO
from ultralytics.engine.exporter import NMSModel

torch.cuda.empty_cache()
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
dataset_root = Path(DATA_DIR)
dataset_yaml_path = Path(train_config.DATASET_YAML)

dataset_yaml = {
    "path": str(dataset_root.resolve()),
    "train": "train/images",
    "val": "valid/images",
    "kpt_shape": list(train_config.KPT_SHAPE),
    "names": train_config.CLASSES_DICT,
}

with open(dataset_yaml_path, "w") as f:
    yaml.safe_dump(dataset_yaml, f, sort_keys=False)

print(dataset_yaml_path.read_text())

In [ ]:
train_image_count = len(list(Path(TRAIN_IMG_PATH).glob("*")))
train_label_count = len(list(Path(TRAIN_LABEL_PATH).glob("*.txt")))
valid_image_count = len(list(Path(VALID_IMG_PATH).glob("*")))
valid_label_count = len(list(Path(VALID_LABEL_PATH).glob("*.txt")))

print({
    "train_images": train_image_count,
    "train_labels": train_label_count,
    "valid_images": valid_image_count,
    "valid_labels": valid_label_count,
})

assert train_image_count > 0 and valid_image_count > 0, "No images were found in the train/valid folders."
assert train_image_count == train_label_count, "Train image/label count mismatch."
assert valid_image_count == valid_label_count, "Valid image/label count mismatch."

sample_label = next(Path(TRAIN_LABEL_PATH).glob("*.txt"))
sample_values = sample_label.read_text().strip().split()
expected_values = 1 + 4 + (train_config.KPT_SHAPE[0] * train_config.KPT_SHAPE[1])
print("Sample label:", sample_label.name)
print("Columns in sample label:", len(sample_values), "Expected:", expected_values)
assert len(sample_values) == expected_values, "YOLO pose label format is incorrect."

In [ ]:
device = 0 if torch.cuda.is_available() else "cpu"

model = YOLO(train_config.MODEL)
train_results = model.train(
    data=str(dataset_yaml_path),
    epochs=train_config.EPOCHS,
    imgsz=data_config.IMAGE_SIZE,
    batch=data_config.BATCH_SIZE,
    close_mosaic=data_config.CLOSE_MOSAIC,
    mosaic=data_config.MOSAIC,
    fliplr=data_config.FLIP_LR,
    project=train_config.PROJECT,
    name=train_config.NAME,
    device=device,
    pretrained=True,
    plots=True,
    val=True,
)

run_dir = Path(model.trainer.save_dir)
best_pt = run_dir / "weights" / "best.pt"
last_pt = run_dir / "weights" / "last.pt"

print("Run directory:", run_dir)
print("Best weights:", best_pt)
print("Last weights:", last_pt)
assert best_pt.exists(), f"Trained weights were not found at {best_pt}"

In [ ]:
best_model = YOLO(str(best_pt))
metrics = best_model.val(
    data=str(dataset_yaml_path),
    imgsz=data_config.IMAGE_SIZE,
    device=device,
)

metrics.results_dict

In [ ]:
raw_onnx_path = best_model.export(
    format="onnx",
    imgsz=data_config.IMAGE_SIZE,
    batch=1,
    simplify=True,
    dynamic=False,
    opset=train_config.OPSET,
    nms=False,
)

print("Raw ONNX export:", raw_onnx_path)

In [ ]:
class BrowserFriendlyPoseWrapper(torch.nn.Module):
    """
    Wraps a trained Ultralytics pose model so the exported ONNX output matches
    the browser parser layout:
    [x_center, y_center, width, height, score, kpt1_x, kpt1_y, kpt1_v, ...]
    """

    def __init__(self, yolo_model, conf=0.35, iou=0.45, max_det=25, opset=17):
        super().__init__()
        export_args = SimpleNamespace(
            format="onnx",
            conf=conf,
            iou=iou,
            max_det=max_det,
            agnostic_nms=False,
            dynamic=False,
            batch=1,
            opset=opset,
            int8=False,
        )
        self.nms_model = NMSModel(yolo_model.model, export_args)

    def forward(self, images):
        # dets: [B, max_det, 4 + 2 + K*D] = [x1, y1, x2, y2, score, cls_id, ...]
        dets = self.nms_model(images)

        boxes_xyxy = dets[..., :4]
        scores = dets[..., 4:5]
        keypoints = dets[..., 6:]  # drop class_id so the website parser matches exactly

        x1 = boxes_xyxy[..., 0:1]
        y1 = boxes_xyxy[..., 1:2]
        x2 = boxes_xyxy[..., 2:3]
        y2 = boxes_xyxy[..., 3:4]

        xywh = torch.cat(
            (
                (x1 + x2) / 2.0,
                (y1 + y2) / 2.0,
                (x2 - x1).clamp(min=0),
                (y2 - y1).clamp(min=0),
            ),
            dim=-1,
        )
        return torch.cat((xywh, scores, keypoints), dim=-1)

In [ ]:
best_model.model = best_model.model.float().eval().cpu()

browser_wrapper = BrowserFriendlyPoseWrapper(
    best_model,
    conf=train_config.CONF_THRESHOLD,
    iou=train_config.IOU_THRESHOLD,
    max_det=train_config.MAX_DET,
    opset=train_config.OPSET,
).eval().cpu()

browser_models_dir = Path("models")
browser_models_dir.mkdir(parents=True, exist_ok=True)

browser_onnx_path = browser_models_dir / train_config.WEB_ONNX_NAME
dummy_input = torch.zeros(1, 3, data_config.IMAGE_SIZE, data_config.IMAGE_SIZE, dtype=torch.float32)

torch.onnx.export(
    browser_wrapper,
    dummy_input,
    str(browser_onnx_path),
    export_params=True,
    opset_version=train_config.OPSET,
    do_constant_folding=True,
    input_names=["images"],
    output_names=["output0"],
)

print("Browser ONNX export:", browser_onnx_path.resolve())

In [ ]:
try:
    from onnxsim import simplify

    browser_model_onnx = onnx.load(str(browser_onnx_path))
    simplified_model, check = simplify(browser_model_onnx)

    if check:
        onnx.save(simplified_model, str(browser_onnx_path))
        print("Simplified browser ONNX model.")
    else:
        print("ONNX simplification check returned False, keeping the original export.")
except Exception as e:
    print("Skipping ONNX simplification:", e)

onnx_model = onnx.load(str(browser_onnx_path))
onnx.checker.check_model(onnx_model)

def _shape(io_node):
    dims = []
    for dim in io_node.type.tensor_type.shape.dim:
        if dim.dim_value:
            dims.append(dim.dim_value)
        else:
            dims.append(dim.dim_param)
    return dims

print("Inputs:")
for node in onnx_model.graph.input:
    print(f"  - {node.name}: {_shape(node)}")

print("Outputs:")
for node in onnx_model.graph.output:
    print(f"  - {node.name}: {_shape(node)}")

print("Website-ready model saved to:", browser_onnx_path.resolve())